# Kenyan Personal Expense Analyzer

Explore a fictional month of personal spending in Kenyan shillings (KES) using pandas and Matplotlib. The analysis summarizes spending by category, date, and payment method, then exports reusable summaries.

**Limitations:** The sample data is fictional, covers one month only, and contains no income or budget information. It should not be treated as a complete picture of a person's finances.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Expense Analyser is ready!")

## Dataset

Each row is one fictional transaction. `Date` is the transaction date, `Category` is the spending category, `Amount_KES` is the amount in Kenyan shillings, and `Payment_Method` is how it was paid.

In [ ]:
data = {
    "Date": [
        "2026-09-01", "2026-09-02", "2026-09-02",
        "2026-09-03", "2026-09-04", "2026-09-05",
        "2026-09-06", "2026-09-07", "2026-09-08",
        "2026-09-09", "2026-09-10", "2026-09-11",
        "2026-09-12", "2026-09-13", "2026-09-14",
        "2026-09-15", "2026-09-16", "2026-09-17",
        "2026-09-18", "2026-09-19", "2026-09-20",
        "2026-09-21", "2026-09-22", "2026-09-23",
        "2026-09-24", "2026-09-25", "2026-09-26",
        "2026-09-27", "2026-09-28", "2026-09-29"
    ],
    "Category": [
        "Food", "Transport", "Food", "Entertainment", "Rent",
        "Food", "Transport", "Shopping", "Food", "Airtime",
        "Transport", "Food", "Entertainment", "Food", "Rent",
        "Transport", "Shopping", "Food", "Airtime", "Transport",
        "Food", "Entertainment", "Shopping", "Food", "Transport",
        "Food", "Airtime", "Transport", "Food", "Entertainment"
    ],
    "Amount_KES": [
        250, 150, 400, 500, 8000,
        300, 200, 1200, 350, 100,
        180, 450, 300, 250, 8000,
        150, 900, 400, 100, 200,
        350, 450, 700, 300, 180,
        500, 100, 150, 350, 250
    ],
    "Payment_Method": [
        "M-Pesa", "Cash", "M-Pesa", "M-Pesa", "Bank",
        "Cash", "M-Pesa", "M-Pesa", "Cash", "Airtime",
        "M-Pesa", "Cash", "M-Pesa", "M-Pesa", "Bank",
        "Cash", "M-Pesa", "Cash", "Airtime", "M-Pesa",
        "Cash", "M-Pesa", "M-Pesa", "Cash", "M-Pesa",
        "Cash", "Airtime", "Cash", "M-Pesa", "M-Pesa"
    ]
}

df = pd.DataFrame(data)
df.head()

In [ ]:
from io import BytesIO
from pathlib import Path
import re

from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
except ImportError as error:
    raise ImportError(
        "The upload panel requires ipywidgets. Install it with: pip install ipywidgets"
    ) from error


display(Markdown(
    "## Upload receipts and statements\n\n"
    "Choose CSV statement files and receipt images or PDFs, then select **Import files**. "
    "Statement CSVs need a date column and an amount column; category and payment method "
    "are optional. Missing categories are marked Uncategorized. Receipt images and PDFs are "
    "saved in `uploaded_receipts/` and are not OCR-processed. Use the entry fields below "
    "to add a transaction after checking a receipt. Files stay in this project folder, so "
    "avoid uploading sensitive documents to a shared workspace."
))

uploader = widgets.FileUpload(
    accept=".csv,.png,.jpg,.jpeg,.webp,.pdf",
    multiple=True,
    description="Choose files",
)
import_button = widgets.Button(
    description="Import files",
    icon="upload",
    button_style="primary",
)
upload_status = widgets.Output()

receipt_date = widgets.DatePicker(description="Date")
receipt_category = widgets.Text(description="Category", placeholder="e.g. Food")
receipt_amount = widgets.FloatText(description="Amount (KES)", min=0)
receipt_payment = widgets.Text(description="Payment", value="Cash")
add_receipt_button = widgets.Button(
    description="Add receipt transaction",
    icon="plus",
)
receipt_status = widgets.Output()


def _uploaded_items(value):
    if isinstance(value, dict):
        return [dict(details, name=name) for name, details in value.items()]
    return list(value)


def _column_key(column):
    return re.sub(r"[^a-z0-9]+", "_", str(column).strip().lower()).strip("_")


def _statement_transactions(content):
    statement = pd.read_csv(BytesIO(bytes(content)), encoding="utf-8-sig", sep=None, engine="python")
    columns = {_column_key(column): column for column in statement.columns}
    date_column = next(
        (columns[name] for name in ("date", "transaction_date", "value_date") if name in columns),
        None,
    )
    amount_column = next(
        (
            columns[name]
            for name in ("amount_kes", "amount", "transaction_amount", "debit", "withdrawal", "paid_out")
            if name in columns
        ),
        None,
    )
    if date_column is None or amount_column is None:
        raise ValueError("CSV needs a date column and an amount, debit, or withdrawal column.")

    category_column = next(
        (columns[name] for name in ("category", "type") if name in columns),
        None,
    )
    payment_column = next(
        (columns[name] for name in ("payment_method", "payment", "channel") if name in columns),
        None,
    )
    transactions = pd.DataFrame({
        "Date": statement[date_column],
        "Amount_KES": pd.to_numeric(
            statement[amount_column]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace(r"[^0-9.\\-]", "", regex=True),
            errors="coerce",
        ),
        "Category": statement[category_column] if category_column else "Uncategorized",
        "Payment_Method": statement[payment_column] if payment_column else "Statement",
    })
    transactions["Category"] = transactions["Category"].fillna("Uncategorized").replace("", "Uncategorized")
    transactions["Payment_Method"] = transactions["Payment_Method"].fillna("Statement").replace("", "Statement")
    return transactions


def _save_receipt(filename, content):
    receipt_dir = Path("uploaded_receipts")
    receipt_dir.mkdir(exist_ok=True)
    safe_name = Path(filename).name
    destination = receipt_dir / safe_name
    suffix = 1
    while destination.exists():
        destination = receipt_dir / f"{Path(safe_name).stem}_{suffix}{Path(safe_name).suffix}"
        suffix += 1
    destination.write_bytes(bytes(content))
    return destination


def _import_files(_):
    global df
    imported_statements = []
    saved_receipts = 0
    errors = []
    with upload_status:
        upload_status.clear_output()
        for item in _uploaded_items(uploader.value):
            filename = item.get("name", "")
            extension = Path(filename).suffix.lower()
            try:
                if extension == ".csv":
                    imported_statements.append(_statement_transactions(item["content"]))
                elif extension in {".png", ".jpg", ".jpeg", ".webp", ".pdf"}:
                    _save_receipt(filename, item["content"])
                    saved_receipts += 1
                else:
                    errors.append(f"{filename}: unsupported file type")
            except Exception as error:
                errors.append(f"{filename}: {error}")

        if imported_statements:
            df = pd.concat([df, *imported_statements], ignore_index=True)
        print(
            f"Added {sum(len(frame) for frame in imported_statements)} statement transactions; "
            f"saved {saved_receipts} receipt file(s)."
        )
        for message in errors:
            print("Skipped:", message)
        if not _uploaded_items(uploader.value):
            print("Choose one or more files first.")


def _add_receipt_transaction(_):
    global df
    with receipt_status:
        receipt_status.clear_output()
        category = receipt_category.value.strip()
        if receipt_date.value is None or not category or receipt_amount.value < 0:
            print("Enter a date, category, and non-negative amount.")
            return
        transaction = pd.DataFrame([{
            "Date": receipt_date.value.isoformat(),
            "Category": category,
            "Amount_KES": receipt_amount.value,
            "Payment_Method": receipt_payment.value.strip() or "Cash",
        }])
        df = pd.concat([df, transaction], ignore_index=True)
        print("Receipt transaction added to the analysis.")


import_button.on_click(_import_files)
add_receipt_button.on_click(_add_receipt_transaction)
display(uploader, import_button, upload_status)
display(widgets.HBox([receipt_date, receipt_category]))
display(widgets.HBox([receipt_amount, receipt_payment, add_receipt_button]))
display(receipt_status)

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns)
print("\nData types:")
print(df.dtypes)
print("\nDataset information:")
df.info()

## Cleaning

Parse dates and amounts, check missing values and duplicates, then remove duplicate rows, invalid essential values, and negative expenses. Zero-value expenses are retained.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Amount_KES"] = pd.to_numeric(df["Amount_KES"], errors="coerce")

print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()
df = df.dropna(subset=["Date", "Category", "Amount_KES"])
df = df[df["Amount_KES"] >= 0].copy()

print("\nCleaned dataset shape:", df.shape)

## Overall spending

Calculate a few basic measures to describe the cleaned transactions.

In [ ]:
total_spent = df["Amount_KES"].sum()
average_transaction = df["Amount_KES"].mean()
largest_transaction = df["Amount_KES"].max()
smallest_transaction = df["Amount_KES"].min()
number_of_transactions = len(df)

print(f"Total spent: KSh {total_spent:,.2f}")
print(f"Average transaction: KSh {average_transaction:,.2f}")
print(f"Largest transaction: KSh {largest_transaction:,.2f}")
print(f"Smallest transaction: KSh {smallest_transaction:,.2f}")
print(f"Number of transactions: {number_of_transactions}")

In [ ]:
category_spending = (
    df.groupby("Category")["Amount_KES"]
      .sum()
      .sort_values(ascending=False)
)

category_share = (category_spending / category_spending.sum() * 100).round(2)
category_summary = pd.DataFrame({
    "Total_KES": category_spending,
    "Share_Percent": category_share
})

category_summary

In [ ]:
category_spending.plot(
    kind="bar",
    figsize=(9, 5),
    title="Spending by Category"
)
plt.xlabel("Category")
plt.ylabel("Amount (KES)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
category_spending.plot(
    kind="pie",
    figsize=(7, 7),
    autopct="%1.1f%%",
    startangle=90,
    ylabel="",
    title="Share of Total Spending"
)
plt.tight_layout()
plt.show()

In [ ]:
daily_spending = (
    df.groupby("Date")["Amount_KES"]
      .sum()
      .sort_index()
)

daily_spending.plot(
    kind="line",
    marker="o",
    figsize=(11, 5),
    title="Daily Spending Trend"
)
plt.xlabel("Date")
plt.ylabel("Amount (KES)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

highest_spending_day = daily_spending.idxmax()
highest_daily_amount = daily_spending.max()
print("Highest-spending day:", highest_spending_day.date())
print(f"Amount spent that day: KSh {highest_daily_amount:,.2f}")

In [ ]:
top_category = category_spending.idxmax()
top_category_amount = category_spending.max()
top_category_share = category_share.loc[top_category]

print("=" * 40)
print("PERSONAL EXPENSE ANALYSIS REPORT")
print("=" * 40)
print(f"Total spending: KSh {total_spent:,.2f}")
print(f"Transactions analysed: {len(df)}")
print(f"Average transaction: KSh {average_transaction:,.2f}")
print(f"\nHighest-spending category: {top_category}")
print(f"Amount spent in this category: KSh {top_category_amount:,.2f}")
print(f"Share of total spending: {top_category_share:.2f}%")
print(f"\nHighest-spending date: {highest_spending_day.date()}")
print(f"Amount spent that day: KSh {highest_daily_amount:,.2f}")
print("\nRECOMMENDATION")
print(
    f"Review spending in {top_category}, then determine whether it is "
    "necessary, recurring, or potentially reducible."
)

In [ ]:
payment_spending = (
    df.groupby("Payment_Method")["Amount_KES"]
      .sum()
      .sort_values(ascending=False)
)
print(payment_spending)

payment_spending.plot(
    kind="bar",
    figsize=(8, 4),
    title="Spending by Payment Method"
)
plt.xlabel("Payment Method")
plt.ylabel("Amount (KES)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

transaction_count = df["Payment_Method"].value_counts()
print(transaction_count)

## Findings and recommendations

- Compare category totals and shares to identify the largest recorded spending areas.
- Review the daily trend for unusually high-spending dates.
- Compare payment methods by both total amount and transaction count.
- Treat high totals as prompts for review, not automatic evidence that a cost can be reduced; rent and other recurring expenses may be necessary.

The data is fictional, represents only September 2026, and does not include income or a budget. As a result, this analysis cannot estimate savings, affordability, or whether spending is appropriate.

In [ ]:
category_summary.to_csv("category_spending_summary.csv")
daily_spending.to_csv("daily_spending_summary.csv")
print("Analysis files exported successfully!")